In [29]:
import requests as r
from utils import query_raven
import logging
import sys
from llama_index.llms.openai import OpenAI
from llama_index.core.query_engine import JSONalyzeQueryEngine

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))


In [30]:
import os
import openai

# os.environ["OPENAI_API_KEY"] = "sk-proj-eh7lc5VDVDlkfSJ0x4QoT3BlbkFJxxlBBbiYyGLlEfRqzu8M"
openai.api_key = os.environ["OPENAI_API_KEY"]
from IPython.display import Markdown, display

In [31]:

def needs_quotes(field):
    # List of fields that need single quotes
    quoted_fields = ['CustomerID', 'ShipName', 'ShipCountry', 'OrderDate', 'ShipCity']
    print(field)
    return field in quoted_fields


def format_value(parameter, value):
    if needs_quotes(parameter):
        return f"'{value}'"
    return value

def get_one_parameter_and_value(parameter1: str, operator1: str, value1: str):
    value1 = format_value(parameter1, value1)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1}&$format=json"
    print("Generated URL for one parameter:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_two_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2}&$format=json"
    print("Generated URL for two parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_three_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str, parameter3: str, operator3: str, value3: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    value3 = format_value(parameter3, value3)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2} and {parameter3} {operator3} {value3}&$format=json"
    print("Generated URL for three parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()


In [32]:
# USER_QUERY = 'how many orders were sent to Germany were delayed in their delivery'
# USER_QUERY = "How many orders sent to germany have freight cost greater than 20 $"
# USER_QUERY = "How many orders have been handled by employeeID 2. How many of those have been delayed in their delivery?"
# USER_QUERY = 'Which orders to Italy have ben processed by employee ID 4'
# USER_QUERY = 'Shipping Partner Ernst Handel has done how many deliveries to Austria. how many of those orders have been delayed in their delivery ?'
# USER_QUERY = 'What is the average freight cost for orders shipped to France?'
# USER_QUERY = 'How many orders to the Italy have a shipping cost of less than $200?'
USER_QUERY = 'Which orders were shipped using ShipVia ID 1 and have been delayed?'
# USER_QUERY = 'What is the average freight cost of orders shipped to France sent by shipper with ID 2?'
# USER_QUERY = 'How many orders have been handled by employeeID 3? How many of those have been shipped on time'


In [33]:
raven_functions = \
f'''

RULE NO 1: Never send null in values. If there is no value for any parameter, use a function with lesser parameters.
RULE NO 2: Always Select a function depending on the number of parameters to be sent to that function.
RULE NO 3: Use this Function to pass one parameter and its value 
            def get_one_parameter_and_value(parameter1 : str, operator1: str , value1 : str):
RULE NO 4: Use this Function to pass two parameters and its values 
            def get_two_parameters_and_values(parameter1 : str, operator1: str, value1 : str, parameter2 : str, operator2: str, value2 : str):
RULE NO 5: Use this Function to pass three parameters and its values
            def get_three_parameters_and_values(parameter1 : str, operator1: str, value1 : str, parameter2 : str, operator2: str, value2 : str,  parameter3 : str, operator3: str, value3 : str):

Operator values will be ge for Greater than or equal to
                        le for Less than or equal to
                        eq for equal to

The parameter variable can fall under any of these given below. Depending on the question asked you need to extaract relevant categories to be passed on to the API.

    **OrderID:** The unique ID of the order.
    **CustomerID:** The ID of the customer who placed the order.
    **EmployeeID:** The ID of the employee who took the order.
    **OrderDate:** The date the order was placed.
    **RequiredDate:** The date the order is required to be shipped by.
    **ShippedDate:** The date the order was shipped. 
    **ShipVia:** The ID of the shipper used for the order. 
    **Freight:** The shipping cost of the order.
    **ShipName:** The name of the shipping recipient. This filed can be addressed as shipping partner or Shipper.
    **ShipAddress:** The address of the shipping recipient.
    **ShipCity:** The city of the shipping recipient.
    **ShipRegion:** The region of the shipping recipient (e.g., state or province).
    **ShipPostalCode:** The postal code of the shipping recipient.
    **ShipCountry:** The country of the shipping recipient.

    Impoprtant: For delayed delievries check if RequiredDate is less than or equal to ShippedDate .

"""

User Query: {USER_QUERY}<human_end>
    """
'''

call = query_raven(raven_functions)
print(call)

get_two_parameters_and_values(parameter1='ShipVia', operator1='eq', value1='1', parameter2='RequiredDate', operator2='le', value2='ShippedDate')


In [34]:
response = eval(call)

ShipVia
RequiredDate
Generated URL for two parameters: https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipVia eq 1 and RequiredDate le ShippedDate&$format=json


In [35]:
# print(response)

In [36]:
llm = OpenAI(model="gpt-3.5-turbo")

In [37]:
# from langchain_core.messages import HumanMessage, SystemMessage

# messages = [
#     SystemMessage(content="You are a json data analysis expert. You can parse through multiple properties and its values within a json dataset and give meaningful insights from it. You work best when there is only snippets of data and not the whole dataset. Make assumptions where necessary, do the analysis which is asked and then state the assumptions in the end"),
#     HumanMessage(content=f"This json contains orders shipped to France sent by shipper with ID 2 {response}. using this perform the actual calculation, process the numbers and give you the precise average for Freight")
# ]


In [38]:
json_stats_query_engine = JSONalyzeQueryEngine(
    list_of_dict=response['value'],
    llm=llm,
    verbose=True,
)

In [39]:
display(
    # Markdown("> Question: {}".format(USER_QUERY)),
    Markdown("Answer: {}".format(json_stats_query_engine.query(USER_QUERY))),
)

Query: Which orders were shipped using ShipVia ID 1 and have been delayed?
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
SQL Query: SELECT OrderID
FROM items
WHERE ShipVia = 1 AND ShippedDate > RequiredDate;
Table Schema: {'OrderID': <class 'int'>, 'CustomerID': <class 'str'>, 'EmployeeID': <class 'int'>, 'OrderDate': <class 'str'>, 'RequiredDate': <class 'str'>, 'ShippedDate': <class 'str'>, 'ShipVia': <class 'int'>, 'Freight': <class 'float'>, 'ShipName': <class 'str'>, 'ShipAddress': <class 'str'>, 'ShipCity': <class 'str'>, 'ShipRegion': <class 'str'>, 'ShipPostalCode': <class 'str'>, 'ShipCountry': <class 'str'>}
SQL Response: [{'OrderID': 10280}, {'OrderID': 10309}, {'OrderID': 10515}, {'OrderID': 10

Answer: Orders with OrderIDs 10280, 10309, 10515, 10596, 10660, 10726, 10727, 10807, 10828, 10927, 10960, and 10970 were shipped using ShipVia ID 1 and have been delayed.

In [40]:
print("Answer: {}".format(json_stats_query_engine.query(USER_QUERY)))

Query: Which orders were shipped using ShipVia ID 1 and have been delayed?
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
SQL Query: SELECT OrderID
FROM items
WHERE ShipVia = 1
AND ShippedDate > RequiredDate;
Table Schema: {'OrderID': <class 'int'>, 'CustomerID': <class 'str'>, 'EmployeeID': <class 'int'>, 'OrderDate': <class 'str'>, 'RequiredDate': <class 'str'>, 'ShippedDate': <class 'str'>, 'ShipVia': <class 'int'>, 'Freight': <class 'float'>, 'ShipName': <class 'str'>, 'ShipAddress': <class 'str'>, 'ShipCity': <class 'str'>, 'ShipRegion': <class 'str'>, 'ShipPostalCode': <class 'str'>, 'ShipCountry': <class 'str'>}
SQL Response: [{'OrderID': 10280}, {'OrderID': 10309}, {'OrderID': 10515}, {'OrderID': 10

In [41]:
result2=json_stats_query_engine.query(USER_QUERY)

Query: Which orders were shipped using ShipVia ID 1 and have been delayed?
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
SQL Query: SELECT OrderID
FROM items
WHERE ShipVia = 1 AND ShippedDate > RequiredDate;
Table Schema: {'OrderID': <class 'int'>, 'CustomerID': <class 'str'>, 'EmployeeID': <class 'int'>, 'OrderDate': <class 'str'>, 'RequiredDate': <class 'str'>, 'ShippedDate': <class 'str'>, 'ShipVia': <class 'int'>, 'Freight': <class 'float'>, 'ShipName': <class 'str'>, 'ShipAddress': <class 'str'>, 'ShipCity': <class 'str'>, 'ShipRegion': <class 'str'>, 'ShipPostalCode': <class 'str'>, 'ShipCountry': <class 'str'>}
SQL Response: [{'OrderID': 10280}, {'OrderID': 10309}, {'OrderID': 10515}, {'OrderID': 10

In [42]:
result2.response

'Orders with OrderID 10280, 10309, 10515, 10596, 10660, 10726, 10727, 10807, 10828, 10927, 10960, and 10970 were shipped using ShipVia ID 1 and have been delayed.'